<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 8: Zaman Serisi Prophet

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 8 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta08/hafta08_zaman_serisi_prophet.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta08/hafta08_zaman_serisi_prophet.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 8 - Zaman Serisi Tahmini (Prophet)

Bu derste:
- Facebook Prophet kütüphanesini kullanarak zaman serisi tahmini yapacağız
- Sentetik satış verisi oluşturacağız (trend + mevsimsellik)
- Model eğitip 90 günlük tahmin yapacağız
- Trend, haftalık ve yıllık bileşenleri inceleyeceğiz
- Çapraz doğrulama ile model performansını değerlendireceğiz

## 1. Kütüphanelerin Kurulumu ve Yüklenmesi

### Gerekli Paketlerin Kurulumu

Aşağıdaki komut ile ihtiyaç duyulan Python paketlerini yüklüyoruz.

In [ ]:
!pip install prophet -q

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `prophet` | Facebook Prophet: zaman serisi tahmini |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric

plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

## 2. Gerçek Zaman Serisi Verisini Yükleme

**Airline Passengers** — 1949-1960 yılları arası uluslararası havayolu yolcu sayıları. Klasik zaman serisi analizi referans veri setidir. Belirgin trend ve mevsimsellik içerir.

**Kaynak:** [Box & Jenkins (1976)](https://www.kaggle.com/datasets/rakannimer/air-passengers)

In [ ]:
# Airline Passengers Dataset (gerçek veri)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df_raw = pd.read_csv(url, names=['ds', 'y'], header=0)
df_raw['ds'] = pd.to_datetime(df_raw['ds'])

print(f"Veri seti boyutu: {df_raw.shape}")
print(f"Zaman aralığı: {df_raw['ds'].min().strftime('%Y-%m')} → {df_raw['ds'].max().strftime('%Y-%m')}")
print(f"\nYolcu sayısı (bin):")
print(f"  Min: {df_raw['y'].min()}")
print(f"  Max: {df_raw['y'].max()}")
print(f"  Ortalama: {df_raw['y'].mean():.0f}")

# Görselleştirelim
plt.figure(figsize=(14, 6))
plt.plot(df_raw['ds'], df_raw['y'], 'b-', linewidth=2)
plt.title('Uluslararası Havayolu Yolcu Sayıları (1949-1960)', fontsize=15)
plt.xlabel('Tarih')
plt.ylabel('Yolcu Sayısı (Bin)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Prophet için hazır format
prophet_df = df_raw.copy()
prophet_df.head()

## 3. Veriyi Görselleştirme

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['ds'], df['y'], color='steelblue', alpha=0.7, linewidth=0.8)
plt.xlabel('Tarih')
plt.ylabel('Günlük Satış (TL)')
plt.title('Günlük Satış Verisi (2 Yıl)')
plt.grid(True, alpha=0.3)
plt.show()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Aylık ortalama satış
monthly = df.set_index('ds').resample('M')['y'].mean()

plt.figure(figsize=(14, 5))
monthly.plot(kind='bar', color='seagreen', alpha=0.8)
plt.xlabel('Ay')
plt.ylabel('Ortalama Günlük Satış (TL)')
plt.title('Aylık Ortalama Satış')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Prophet Modeli Eğitme

Prophet, Facebook tarafından geliştirilen bir zaman serisi tahmin kütüphanesidir.

**Önemli:** Prophet `ds` (tarih) ve `y` (değer) sütunlarını gerektirir.

In [ ]:
# Prophet modelini oluştur ve eğit
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)

model.fit(prophet_df)
print("Model eğitimi tamamlandı!")

## 5. Gelecek Tahmini (90 Gün)

### 90 günlük gelecek veri çerçevesi oluştur

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# 90 günlük gelecek veri çerçevesi oluştur
future = model.make_future_dataframe(periods=90)
print(f"Gelecek veri çerçevesi boyutu: {future.shape}")
print(f"Son tahmin tarihi: {future['ds'].max()}")
future.tail()

### Streamlit Uygulaması

Streamlit ile interaktif bir veri uygulaması oluşturuyoruz. Streamlit, Python kodunu otomatik olarak web arayüzüne dönüştürür.

In [ ]:
# Tahmin yap
forecast = model.predict(future)
print("Tahmin sütunları:")
print(forecast.columns.tolist())
print(f"\nTahmin edilen son 5 gün:")
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

## 6. Tahmin Görselleştirme

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Prophet'in yerleşik grafiği
fig1 = model.plot(forecast)
plt.title('Satış Tahmini (Prophet)', fontsize=14)
plt.xlabel('Tarih')
plt.ylabel('Günlük Satış (TL)')
plt.show()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Özel görselleştirme
plt.figure(figsize=(14, 7))

# Gerçek veriler
plt.plot(df['ds'], df['y'], 'b.', alpha=0.3, label='Gerçek Veriler', markersize=3)

# Tahmin
forecast_future = forecast[forecast['ds'] > df['ds'].max()]
plt.plot(forecast_future['ds'], forecast_future['yhat'], 'r-', 
         linewidth=2, label='Tahmin')
plt.fill_between(forecast_future['ds'], 
                 forecast_future['yhat_lower'], 
                 forecast_future['yhat_upper'],
                 color='red', alpha=0.15, label='Güven Aralığı')

plt.axvline(x=df['ds'].max(), color='gray', linestyle='--', alpha=0.5, label='Tahmin Başlangıcı')
plt.xlabel('Tarih', fontsize=12)
plt.ylabel('Günlük Satış (TL)', fontsize=12)
plt.title('90 Günlük Satış Tahmini', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 7. Bileşen Analizi (Trend, Haftalık, Yıllık)

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Prophet'in bileşen grafiği
fig2 = model.plot_components(forecast)
plt.show()

### Bileşen Yorumları

- **Trend**: Satışlarda sürekli yükselen bir trend görülüyor
- **Haftalık**: Hafta sonları satışlar belirgin şekilde artıyor
- **Yıllık**: Yaz aylarında (Haziran-Ağustos) satışlar en yüksek seviyede, kış aylarında düşüyor

## 8. Çapraz Doğrulama (Cross Validation)

Model performansını değerlendirmek için zaman serisi çapraz doğrulama yapıyoruz.

In [ ]:
# Çapraz doğrulama
# initial: eğitim dönemi, period: tahmin aralığı, horizon: tahmin süresi
df_cv = cross_validation(
    model, 
    initial='365 days', 
    period='30 days', 
    horizon='30 days'
)
df_cv.head()

### Performans metrikleri

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Performans metrikleri
df_p = performance_metrics(df_cv)
print("Performans Metrikleri:")
df_p[['horizon', 'mse', 'rmse', 'mae', 'mape']].head(10)

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# MAPE görselleştirme
fig3 = plot_cross_validation_metric(df_cv, metric='mape')
plt.title('MAPE (Ortalama Mutlak Yüzde Hata)')
plt.show()

## 9. Tatil/Özel Gün Etkisi (Bonus)

Prophet'e özel günler ekleyerek modeli geliştirebiliriz.

In [ ]:
# Türkiye'deki önemli günler
holidays = pd.DataFrame({
    'holiday': 'ozel_gun',
    'ds': pd.to_datetime([
        '2024-01-01', '2024-04-23', '2024-05-01', '2024-05-19',
        '2024-07-15', '2024-08-30', '2024-10-29',
        '2025-01-01', '2025-04-23', '2025-05-01', '2025-05-19',
        '2025-07-15', '2025-08-30', '2025-10-29'
    ]),
    'lower_window': 0,
    'upper_window': 1,
})

model_with_holidays = Prophet(
    holidays=holidays,
    yearly_seasonality=True,
    weekly_seasonality=True
)
model_with_holidays.fit(df)
print("Tatil eklenmiş model eğitildi!")

## Özet

Bu derste öğrendiklerimiz:
- **Prophet**, zaman serisi tahmininde güçlü ve kullanımı kolay bir araçtır
- Veriyi `ds` ve `y` sütunlarıyla hazırlamak yeterlidir
- **Bileşen analizi** ile trend, mevsimsellik ve tatil etkilerini ayrıştırabiliriz
- **Çapraz doğrulama** ile model performansını güvenilir şekilde ölçebiliriz
- Tatil ve özel gün bilgisi ekleyerek tahmin kalitesini artırabiliriz

### Alıştırma
1. `changepoint_prior_scale` parametresini değiştirerek trendin esnekliğini ayarlayın
2. 180 günlük tahmin yapın ve sonuçları karşılaştırın
3. Farklı mevsimsellik parametreleri deneyin (Fourier order)

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>